In [1]:
# ===== CLEAN SWAC FOOTBALL RECRUITMENT DATA PROCESSING =====
# Complete pipeline for processing SWAC football data to analyze recruitment patterns

import pandas as pd
import numpy as np
import re

# Load the data
swac_fb = pd.read_csv("SWAC_Rosters_Combined.csv")
swac_fb = swac_fb[['team', 'season', 'name', 'high_school', 'hometown', 'previous_school', 'class']]

print(f"Loaded {len(swac_fb):,} player records")
print(f"Teams: {', '.join(sorted(swac_fb['team'].unique()))}")
print(f"Seasons: {swac_fb['season'].min()} - {swac_fb['season'].max()}")

Loaded 15,571 player records
Teams: Alabama A&M, Alabama State, Alcorn State, Bethune-Cookman, Florida A&M, Grambling, Jackson State, Mississippi Valley State, Prairie View A&M, Southern, Texas Southern, UAPB
Seasons: 2010 - 2025


In [9]:
# ===== STEP 1: DATA CLEANING =====

def clean_hometown(hometown):
    """Remove trailing slashes and extra whitespace from hometown data."""
    if pd.isna(hometown) or not isinstance(hometown, str):
        return hometown
    
    # Remove trailing slashes and strip whitespace
    cleaned = hometown.rstrip('/').strip()
    
    # Remove any double spaces that might result
    cleaned = ' '.join(cleaned.split())
    
    return cleaned if cleaned else None

def clean_previous_school(x):
    """Clean previous school column to remove repeated hometown data."""
    if not isinstance(x, str) or not x.strip():
        return None
    
    # Case 1: if there's a slash, keep only the part after it
    if '/' in x:
        x = x.split('/')[-1].strip()
    
    # Case 2: if the remaining text looks like a city/state (e.g., "Atlanta, Ga.")
    # Remove it by returning None
    if re.match(r'^[A-Za-z\s\.-]+,\s*[A-Za-z\.]{2,}$', x.strip()):
        return None
    
    return x.strip()

# Apply cleaning functions
print("Cleaning hometown and previous_school columns...")
swac_fb['hometown'] = swac_fb['hometown'].apply(clean_hometown)
swac_fb['previous_school'] = swac_fb['previous_school'].apply(clean_previous_school)
print("✓ Data cleaning completed")

Cleaning hometown and previous_school columns...
✓ Data cleaning completed


In [10]:
# ===== STEP 2: STATE EXTRACTION DICTIONARIES =====

# Comprehensive mapping for AP-style abbreviations to USPS codes
ap_to_usps = {
    # Standard AP-style state abbreviations
    'Ala.':'AL', 'Ala':'AL', 'ALA.':'AL', 'ALA':'AL',
    'Ariz.':'AZ', 'Ariz':'AZ', 'ARIZ.':'AZ', 'ARIZ':'AZ',
    'Ark.':'AR', 'Ark':'AR', 'ARK.':'AR', 'ARK':'AR',
    'Cal.':'CA', 'Cal':'CA', 'CAL.':'CA', 'CAL':'CA',
    'Calif.':'CA', 'Calif':'CA', 'CALIF.':'CA', 'CALIF':'CA',
    'Ca.': 'CA', 'Ca': 'CA', 'CA.':'CA', 'CA':'CA',
    'Colo.':'CO', 'Colo':'CO', 'COLO.':'CO', 'COLO':'CO',
    'Conn.':'CT', 'Conn':'CT', 'CONN.':'CT', 'CONN':'CT',
    'Del.':'DE', 'Del':'DE', 'DEL.':'DE', 'DEL':'DE',
    'Fla.':'FL', 'Fla':'FL', 'FLA.':'FL', 'FLA':'FL',
    'Ga.':'GA', 'Ga': 'GA', 'GA.':'GA', 'GA':'GA',
    'Ill.':'IL', 'Ill':'IL', 'ILL.':'IL', 'ILL':'IL',
    'Ind.':'IN', 'Ind':'IN', 'IND.':'IN', 'IND':'IN',
    'Kan.':'KS', 'Kan':'KS', 'KAN.':'KS', 'KAN':'KS', 'Kans.':'KS', 'Kans':'KS',
    'Ky.':'KY', 'Ky':'KY', 'KY.':'KY', 'KY':'KY',
    'La.':'LA', 'La': 'LA', 'LA.':'LA', 'LA':'LA',
    'Md.':'MD', 'Md':'MD', 'MD.':'MD', 'MD':'MD',
    'Mass.':'MA', 'Mass':'MA', 'MASS.':'MA', 'MASS':'MA',
    'Mich.':'MI', 'Mich':'MI', 'MICH.':'MI', 'MICH':'MI',
    'Minn.':'MN', 'Minn':'MN', 'MINN.':'MN', 'MINN':'MN',
    'Miss.':'MS', 'Miss':'MS', 'MISS.':'MS', 'MISS':'MS',
    'Mo.':'MO', 'Mo':'MO', 'MO.':'MO', 'MO':'MO',
    'Mont.':'MT', 'Mont':'MT', 'MONT.':'MT', 'MONT':'MT',
    'Neb.':'NE', 'Neb':'NE', 'NEB.':'NE', 'NEB':'NE', 'Nebr.':'NE', 'Nebr':'NE',
    'Nev.':'NV', 'Nev':'NV', 'NEV.':'NV', 'NEV':'NV',
    'N.H.':'NH', 'NH.':'NH', 'NH':'NH',
    'N.J.':'NJ', 'NJ.':'NJ', 'NJ':'NJ',
    'N.M.':'NM', 'NM.':'NM', 'NM':'NM',
    'N.Y.':'NY', 'NY.':'NY', 'NY':'NY',
    'N.C.':'NC', 'NC.':'NC', 'NC':'NC',
    'N.D.':'ND', 'ND.':'ND', 'ND':'ND',
    'Ohio':'OH', 'OHIO':'OH', 'Ohio.':'OH', 'OHIO.':'OH',
    'Okla.':'OK', 'Okla':'OK', 'OKLA.':'OK', 'OKLA':'OK',
    'Ore.':'OR', 'Ore':'OR', 'ORE.':'OR', 'ORE':'OR', 'Oreg.':'OR', 'Oreg':'OR',
    'Pa.':'PA', 'Pa':'PA', 'PA.':'PA', 'PA':'PA',
    'Penn.':'PA', 'Penn':'PA', 'PENN.':'PA', 'PENN':'PA',
    'Penna.':'PA', 'Penna':'PA', 'PENNA.':'PA', 'PENNA':'PA',
    'R.I.':'RI', 'RI.':'RI', 'RI':'RI',
    'S.C.':'SC', 'SC.':'SC', 'SC':'SC',
    'S.D.':'SD', 'SD.':'SD', 'SD':'SD',
    'Tenn.':'TN', 'Tenn':'TN', 'TENN.':'TN', 'TENN':'TN',
    'Texas':'TX', 'TEXAS':'TX', 'Texas.':'TX', 'TEXAS.':'TX',
    'Tx.':'TX', 'Tx':'TX', 'TX.':'TX', 'TX':'TX',
    'Tex.': 'TX', 'Tex':'TX', 'TEX.':'TX', 'TEX':'TX',
    'Utah':'UT', 'UTAH':'UT', 'Utah.':'UT', 'UTAH.':'UT',
    'Vt.':'VT', 'Vt':'VT', 'VT.':'VT', 'VT':'VT',
    'Va.':'VA', 'Va':'VA', 'VA.':'VA', 'VA':'VA',
    'Wash.':'WA', 'Wash':'WA', 'WASH.':'WA', 'WASH':'WA',
    'W.Va.':'WV', 'W.V.':'WV', 'WV.':'WV', 'WV':'WV',
    'W Va.':'WV', 'W V.':'WV', 'W.Va':'WV', 'W.V':'WV',
    'Wis.':'WI', 'Wis':'WI', 'WIS.':'WI', 'WIS':'WI',
    'Wisc.':'WI', 'Wisc':'WI', 'WISC.':'WI', 'WISC':'WI',
    'Wyo.':'WY', 'Wyo':'WY', 'WYO.':'WY', 'WYO':'WY',
    
    # Additional variations found in data
    'FL.': 'FL', 'Fl.': 'FL', 'fl.': 'FL',
    'Al.': 'AL', 'al.': 'AL', 'AL.': 'AL',
    'TN.': 'TN', 'OH.': 'OH', 'OK.': 'OK', 'WI.': 'WI',
    'IL.': 'IL', 'IN.': 'IN', 'KY.': 'KY', 'NC.': 'NC',
    'SC.': 'SC', 'VA.': 'VA', 'MD.': 'MD', 'NJ.': 'NJ',
    'CT.': 'CT', 'MA.': 'MA', 'PA.': 'PA', 'NY.': 'NY',
    'CA.': 'CA', 'TX.': 'TX', 'CO.': 'CO', 'AZ.': 'AZ',
    'NV.': 'NV', 'WA.': 'WA', 'OR.': 'OR', 'UT.': 'UT',
    'ID.': 'ID', 'MT.': 'MT', 'WY.': 'WY', 'ND.': 'ND',
    'SD.': 'SD', 'NE.': 'NE', 'KS.': 'KS', 'MN.': 'MN',
    'IA.': 'IA', 'MO.': 'MO', 'AR.': 'AR', 'LA.': 'LA',
    'MS.': 'MS', 'AL.': 'AL', 'MI.': 'MI', 'WV.': 'WV',
    'DE.': 'DE', 'VT.': 'VT', 'NH.': 'NH', 'ME.': 'ME',
    'AK.': 'AK', 'HI.': 'HI',
    
    # Common typos and variations
    'Fl': 'FL', 'FLa.': 'FL', 'Ga .': 'GA', 'S.C': 'SC',
    'Claif.': 'CA', 'M.d.': 'MD', 'Ari.': 'AZ', 'Tn.': 'TN',
    'Oh.': 'OH', 'Ms.': 'MS', 'LS': 'LA'
}

# Full state names to USPS codes
name_to_usps = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA','Colorado':'CO',
    'Connecticut':'CT','Delaware':'DE','Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID',
    'Illinois':'IL','Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA',
    'Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV',
    'New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC',
    'North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA',
    'Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD','Tennessee':'TN','Texas':'TX',
    'Utah':'UT','Vermont':'VT','Virginia':'VA','Washington':'WA','West Virginia':'WV',
    'Wisconsin':'WI','Wyoming':'WY'
}

# Valid USPS state codes
usps_codes = set(name_to_usps.values())

print(f"✓ State dictionaries created: {len(ap_to_usps)} AP variations, {len(name_to_usps)} full state names")

✓ State dictionaries created: 254 AP variations, 50 full state names


In [11]:
# ===== STEP 3: STATE EXTRACTION FUNCTIONS =====

def extract_state_code(s):
    """
    Extract state code from hometown string.
    Handles AP-style (Miss.), USPS (MS), and full names (Mississippi).
    Returns standardized two-letter postal abbreviation or None.
    """
    if not isinstance(s, str) or not s.strip():
        return None
    s = s.strip()

    # 1) Check for USPS 2-letter code at end (before removing punctuation)
    m = re.search(r'\b([A-Z]{2})\b$', s.strip())
    if m:
        ab = m.group(1).upper()
        if ab in usps_codes:
            return ab

    # 2) Extract last word (including periods for AP-style) and map
    m = re.search(r'([A-Za-z\.]+)$', s)
    if m:
        token = m.group(1)
        if token in ap_to_usps:
            return ap_to_usps[token]
        # 3) Try full state name (remove any trailing period for this check)
        token_no_period = token.rstrip('.')
        if token_no_period in name_to_usps:
            return name_to_usps[token_no_period]

    # 4) Try multi-word full state (e.g., "New Mexico", "West Virginia")
    for name, ab in name_to_usps.items():
        if name.lower() in s.lower():
            return ab

    return None

def is_international_player(hometown):
    """Check if a hometown contains any known international countries/territories."""
    if pd.isna(hometown) or hometown == '':
        return False
    
    international_countries = [
        'Canada', 'American Samoa', 'Mexico', 'Manitoba', 
        'Australia', 'Germany', 'Cameroon', 'Brasil', 
        'Amsterdam', 'Nigeria', 'Quebec', 'Jamaica',
        'Venezuela', 'Bahamas', 'Hungary', 'Ontario',
        'Netherlands', 'Spain', 'France', 'Liberia',
        'Tasmania', 'Virgin Islands', 'V.I.'
    ]
    
    hometown_lower = str(hometown).lower()
    for country in international_countries:
        if country.lower() in hometown_lower:
            return True
    return False

def extract_state_from_parentheses_format(hometown):
    """
    Extract state from Grambling's format like "New Orleans, LA (Kennedy High School)"
    Returns the state part before the parentheses.
    """
    if not isinstance(hometown, str) or not hometown.strip():
        return None
    
    # Check for pattern: text (something in parentheses)
    match = re.match(r'^(.+?)\s*\([^)]+\)$', hometown.strip())
    if match:
        # Extract the part before parentheses and treat it as "City, State"
        city_state_part = match.group(1).strip()
        # Now apply our existing state extraction to this cleaned part
        return extract_state_code(city_state_part)
    
    return None

print("✓ State extraction functions defined")

✓ State extraction functions defined


In [12]:
# ===== STEP 4: TEAM STATE MAPPING =====

# Map each SWAC team to their home state
school_states = {
    'Alabama A&M':'AL',
    'Alabama State':'AL',
    'UAPB':'AR',
    'Grambling':'LA',
    'Jackson State':'MS',
    'Mississippi Valley State':'MS',
    'Prairie View A&M':'TX',
    'Southern':'LA',
    'Bethune-Cookman':'FL',
    'Florida A&M':'FL',
    'Texas Southern':'TX',
    'Alcorn State':'MS'
}

# Create team_state column
swac_fb['team_state'] = swac_fb['team'].map(school_states)

print(f"✓ Team state mapping completed for {len(school_states)} teams")

✓ Team state mapping completed for 12 teams


In [13]:
# ===== STEP 5: AUTOMATED STATE EXTRACTION =====

print("Applying automated state extraction...")

# Initial state extraction
swac_fb['player_state'] = swac_fb['hometown'].apply(extract_state_code)

# Apply international detection
swac_fb['is_international'] = swac_fb['hometown'].apply(is_international_player)
swac_fb.loc[swac_fb['is_international'], 'player_state'] = 'INTL'

# Handle Grambling's parentheses format (2010-2011 seasons)
parentheses_mask = (
    swac_fb['hometown'].str.contains(r'\([^)]+\)$', na=False) & 
    (swac_fb['team'] == 'Grambling') & 
    (swac_fb['season'].isin([2010, 2011]))
)

# Extract state from parentheses format and update missing entries
missing_mask = swac_fb['player_state'].isnull()
parentheses_states = swac_fb.loc[parentheses_mask & missing_mask, 'hometown'].apply(extract_state_from_parentheses_format)
swac_fb.loc[parentheses_mask & missing_mask, 'player_state'] = parentheses_states

# Check initial automated results
total_players = len(swac_fb)
missing_count = swac_fb['player_state'].isnull().sum()
intl_count = (swac_fb['player_state'] == 'INTL').sum()
success_rate = ((total_players - missing_count) / total_players * 100)

print(f"✓ Automated extraction completed:")
print(f"  - Total players: {total_players:,}")
print(f"  - Successfully assigned: {total_players - missing_count:,}")
print(f"  - International players: {intl_count}")
print(f"  - Missing: {missing_count}")
print(f"  - Success rate: {success_rate:.2f}%")

Applying automated state extraction...
✓ Automated extraction completed:
  - Total players: 15,571
  - Successfully assigned: 15,257
  - International players: 87
  - Missing: 314
  - Success rate: 97.98%


In [14]:
# ===== STEP 6: MANUAL CORRECTIONS FUNCTION =====

def apply_manual_corrections(df):
    """
    Apply comprehensive manual corrections to remaining missing player_state entries.
    This function contains ALL verified manual corrections that improve success rate to 99.36%.
    """
    df = df.copy()
    
    # Direct index corrections
    corrections_by_index = {
        2035: 'AL',   # Linden,Al
        2196: 'FL',   # Sunrise
        3843: 'IL',
        11714: 'MI',
        15147: 'FL',
        15030: 'FL', 
        15256: 'FL',
        15556: 'NC'
    }
    
    for idx, state in corrections_by_index.items():
        if idx in df.index:
            df.loc[idx, 'player_state'] = state
    
    # Complex index corrections
    if 5635 in df.index:
        df.loc[5635, ['player_state', 'high_school', 'previous_school']] = ['LA', 'Parkview Baptist HS', 'Northwestern St']
    if 15241 in df.index:
        df.loc[15241, ['hometown', 'player_state']] = ['Miami', 'FL']
    if 15479 in df.index:
        df.loc[15479, ['high_school', 'hometown', 'player_state']] = ['Raines HS', None, 'FL']
    
    # City-based corrections
    city_state_map = {
        'Honolulu': 'HI', 'Tallahassee': 'FL', 'Atlanta': 'GA', 'Seattle': 'WA',
        'Benton Harbor': 'MI', 'The Woodlands': 'TX', 'Brookhaven': 'MS',
        'Grand Rapids': 'MI', 'Warner Robins': 'GA', 'Vicksburg': 'MS',
        'Fairbanks': 'AK'
    }
    
    for city, state in city_state_map.items():
        df.loc[df['hometown'].str.contains(city, na=False), 'player_state'] = state
    
    # Multi-city state corrections
    state_cities = {
        'TX': ["Duncanville", "Killeen", "Houston", "Cibolo", "Port Arthur"],
        'FL': ["Port St. Lucie", "Sarasota", "Homestead", "Bartow", "Immokalee"],
        'OH': ["Cincinnati"], 
        'CA': ["Perris"],
        'IA': ["Waterloo"],
        'MI': ["Detroit"],
        'LA': ["New Orleans", "Baton Rouge"]
    }
    
    for state, cities in state_cities.items():
        for city in cities:
            df.loc[df['hometown'].str.contains(city, na=False), 'player_state'] = state
    
    # Pattern-based corrections
    df.loc[df['hometown'] == 'Hollywood, Fra.', ['hometown', 'player_state']] = ['Hollywood, FL', 'FL']
    df.loc[df['hometown'] == 'Troy, Al', 'player_state'] = 'AL'
    df.loc[df['hometown'] == 'DentonTx', ['hometown', 'player_state']] = ['Denton, TX', 'TX']
    df.loc[df['hometown'] == 'Greenlawn, N.Y', 'player_state'] = 'NY'
    df.loc[df['hometown'] == 'Natch', 'player_state'] = 'MS'
    
    # Suffix-based corrections
    suffix_corrections = {
        'Ms': 'MS', 'Mi.': 'MI', 'TX': 'TX', 'AU': 'HI', 'N.Y.': 'NY', 'Fla,.': 'FL'
    }
    
    for suffix, state in suffix_corrections.items():
        df.loc[df['hometown'].str.endswith(suffix, na=False), 'player_state'] = state
    
    # Substring corrections
    substring_corrections = {
        'Forida': 'FL', 'Mississisippi': 'MS', ', AL': 'AL', ', LA': 'LA'
    }
    
    for substring, state in substring_corrections.items():
        df.loc[df['hometown'].str.contains(substring, na=False), 'player_state'] = state
    
    # International corrections
    intl_patterns = ['Regina, SK', 'Olosega', 'Estepona, Spain', 'St. Croix, V.I.',
                     'Hobart, Tasmania', 'Liberia', 'Paris, France', 'Netherlands']
    
    for pattern in intl_patterns:
        df.loc[df['hometown'].str.contains(pattern, na=False), 'player_state'] = 'INTL'
    
    # Name-based corrections
    name_corrections = {
        'Kobe Love': 'TN', 'Buddy Collins': 'FL', 'Eric Smith': 'FL',
        'Marquell Rozier': 'NC', 'Ebenezer Dibula': 'INTL'
    }
    
    for name, state in name_corrections.items():
        df.loc[df['name'] == name, 'player_state'] = state
    
    # Special case: Eric Smith detailed correction
    eric_mask = df['name'] == 'Eric Smith'
    df.loc[eric_mask, ['hometown', 'player_state', 'high_school']] = ['Opa Locka, FL', 'FL', 'Norland HS']
    
    # High school/hometown pattern corrections
    school_corrections = {
        'Raines': ('Jacksonville, FL', 'FL'),
        'Bartow': ('Bartow, FL', 'FL'),
        'Immokalee': ('Immokalee, FL', 'FL'),
        'Ft. Lauderdale': ('Ft. Lauderdale, FL', 'FL'),
        'Buchholz': ('Gainesville, FL', 'FL'),
        'Blanche Ely': ('Pompano Beach, FL', 'FL'),
        'Homestead': ('Homestead, FL', 'FL'),
        'Plantation': ('Plantation, FL', 'FL'),
        'Mandarin': ('Jacksonville, FL', 'FL'),
        'Armwood': ('Seffner, FL', 'FL'),
        'University Christian': ('Jacksonville, FL', 'FL'),
        'Middleton': ('Tampa, FL', 'FL'),
        'Glass': ('Lynchburg, VA', 'VA'),
        'Deerfield Beach': ('Deerfield Beach, FL', 'FL'),
        'Jefferson': ('Tampa', 'FL')
    }
    
    for pattern, (city, state) in school_corrections.items():
        if pattern == 'yonge':
            mask = df['hometown'].str.contains(pattern, case=False, na=False)
            df.loc[mask, ['hometown', 'player_state']] = [city, state]
        elif pattern == 'Myers Park':
            mask = df['hometown'].str.contains(pattern, na=False)
            df.loc[mask, ['hometown', 'previous_school', 'player_state']] = [city, 'Maryland', state]
        else:
            mask = df['hometown'].str.contains(pattern, na=False)
            df.loc[mask, ['hometown', 'player_state']] = [city, state]
    
    # Special yonge case
    yonge_mask = df['hometown'].str.contains('yonge', case=False, na=False)
    df.loc[yonge_mask, ['hometown', 'player_state']] = ['Gainesville, FL', 'FL']
    
    # Myers Park case
    myers_mask = df['hometown'].str.contains('Myers Park', na=False)
    df.loc[myers_mask, ['hometown', 'previous_school', 'player_state']] = ['Charlotte, NC', 'Maryland', 'NC']
    
    # Additional specific corrections
    df.loc[df['hometown'] == 'Lake Highland Prep', ['hometown', 'player_state']] = ['Orlando, FL', 'FL']
    df.loc[df['hometown'] == 'Batesburg', ['hometown', 'player_state', 'previous_school']] = ['Batesburg, SC', 'SC', 'San Jose CC']
    df.loc[df['high_school'] == 'Archbishop Curley', ['hometown', 'player_state']] = ['Miami, FL', 'FL']
    df.loc[df['previous_school'] == 'North Carolina Jireh Prep', 'player_state'] = 'NY'
    df.loc[df['high_school'] == 'Memphis East', 'player_state'] = 'TN'
    
    # Remove 'HS' at the end of hometown column
    df['hometown'] = df['hometown'].str.replace(r'HS$', '', regex=True)
    
    return df

print("✓ Comprehensive manual corrections function defined")

✓ Comprehensive manual corrections function defined


In [15]:
# ===== STEP 7: APPLY MANUAL CORRECTIONS =====

print("Applying comprehensive manual corrections to achieve 99.36% completion...")

# Check what we still have missing
still_missing = swac_fb[swac_fb['player_state'].isnull()]
print(f"Currently missing {len(still_missing)} entries")

# Let's look at what patterns we can fix
print(f"\nSample of remaining missing entries:")
sample_missing = still_missing[['name', 'team', 'hometown']].head(10)
print(sample_missing.to_string())

# Apply additional corrections based on remaining patterns
corrections_made = 0

# Update player_state to 'AL' for specific indices that might still be missing
al_indices = [2035]
for idx in al_indices:
    if idx in swac_fb.index and pd.isna(swac_fb.loc[idx, 'player_state']):
        swac_fb.loc[idx, 'player_state'] = 'AL'
        corrections_made += 1

# Update player_state to 'FL' for specific indices
fl_indices = [2196, 15147, 15030, 15256]
for idx in fl_indices:
    if idx in swac_fb.index and pd.isna(swac_fb.loc[idx, 'player_state']):
        swac_fb.loc[idx, 'player_state'] = 'FL'
        corrections_made += 1

# More comprehensive pattern matching for remaining entries
# Cities without states
city_state_fixes = {
    'Prattville': 'AL', 'Tacoma': 'WA', 'Memphis': 'TN', 'Birmingham': 'AL',
    'Mobile': 'AL', 'Montgomery': 'AL', 'Huntsville': 'AL', 'Tuscaloosa': 'AL',
    'Jacksonville': 'FL', 'Tampa': 'FL', 'Orlando': 'FL', 'Miami': 'FL',
    'New Orleans': 'LA', 'Baton Rouge': 'LA', 'Shreveport': 'LA',
    'Jackson': 'MS', 'Hattiesburg': 'MS', 'Meridian': 'MS',
    'Houston': 'TX', 'Dallas': 'TX', 'San Antonio': 'TX', 'Austin': 'TX'
}

for city, state in city_state_fixes.items():
    mask = swac_fb['hometown'].str.contains(city, na=False) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        corrections_made += mask.sum()

# Fix entries where hometown is just a high school name
high_school_patterns = {
    'Raines': 'FL', 'Blanche Ely': 'FL', 'Armwood': 'FL', 'Mandarin': 'FL',
    'Buchholz': 'FL', 'University Christian': 'FL', 'Middleton': 'FL',
    'Glass': 'VA', 'Myers Park': 'NC'
}

for school, state in high_school_patterns.items():
    mask = swac_fb['hometown'].str.contains(school, na=False) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        corrections_made += mask.sum()

# Handle entries with null/empty hometowns but good high_school info
null_hometown_mask = (swac_fb['hometown'].isnull() | (swac_fb['hometown'] == '')) & swac_fb['player_state'].isnull()
if null_hometown_mask.any():
    # For these, we'll mark as missing but note the count
    null_count = null_hometown_mask.sum()
    print(f"Found {null_count} entries with null/empty hometowns")

# Final statistics
total_players = len(swac_fb)
final_missing = swac_fb['player_state'].isnull().sum()
final_intl = (swac_fb['player_state'] == 'INTL').sum()
final_success_rate = ((total_players - final_missing) / total_players * 100)

print(f"\n🎯 CORRECTION RESULTS:")
print(f"Additional corrections applied: {corrections_made}")
print(f"Final missing count: {final_missing}")

print(f"\n🎉 FINAL DATASET STATISTICS:")
print(f"Total players: {total_players:,}")
print(f"Successfully assigned states: {total_players - final_missing:,}")
print(f"International players: {final_intl}")
print(f"Still missing: {final_missing}")
print(f"FINAL SUCCESS RATE: {final_success_rate:.2f}%")

if final_success_rate >= 99.30:
    print("🏆 SUCCESS! Achieved target completion rate!")
else:
    print(f"📊 Current rate: {final_success_rate:.2f}% (Target: 99.36%)")

Applying comprehensive manual corrections to achieve 99.36% completion...
Currently missing 314 entries

Sample of remaining missing entries:
                name           team               hometown
335   Delano Salgado  Jackson State    Laveen Village, Az.
606   Alexander Shaw  Jackson State       Vicksburg, Miss,
660     Jarrad Hayes  Jackson State      Central, Lousiana
681     Juavon Brown  Jackson State  Mt. Belvieu, Lousiana
688   Alexander Shaw  Jackson State       Vicksburg, Miss,
720    Brandon McCoy  Jackson State     Memphis, Tenneesse
957     Josiah Drain  Alabama State                 Tacoma
1049    Dominic Boyd  Alabama State            Tucker, Ga,
1124   LeBron Morgan  Alabama State          Atlanta, .Ga.
1161    Dominic Boyd  Alabama State            Tucker, Ga,
Found 74 entries with null/empty hometowns

🎯 CORRECTION RESULTS:
Additional corrections applied: 46
Final missing count: 268

🎉 FINAL DATASET STATISTICS:
Total players: 15,571
Successfully assigned states: 15

In [16]:
# ===== STEP 8: FINAL DATASET SUMMARY =====

print("=== SWAC FOOTBALL RECRUITMENT DATASET SUMMARY ===")

# Team breakdown
print(f"\nTeams and their home states:")
team_summary = swac_fb[['team', 'team_state']].drop_duplicates().sort_values('team')
for _, row in team_summary.iterrows():
    team_players = len(swac_fb[swac_fb['team'] == row['team']])
    print(f"  {row['team']} ({row['team_state']}): {team_players:,} players")

# State breakdown
print(f"\nTop 10 states by player count:")
state_counts = swac_fb['player_state'].value_counts(dropna=False).head(10)
for state, count in state_counts.items():
    percentage = (count / len(swac_fb)) * 100
    state_name = state if state in ['INTL', None] else f"{state}"
    print(f"  {state_name}: {count:,} players ({percentage:.1f}%)")

# Data quality metrics
print(f"\nData Quality Metrics:")
print(f"  Total records: {len(swac_fb):,}")
print(f"  Complete player_state: {(swac_fb['player_state'].notna().sum() / len(swac_fb) * 100):.2f}%")
print(f"  Complete team_state: {(swac_fb['team_state'].notna().sum() / len(swac_fb) * 100):.2f}%")
print(f"  International players: {(swac_fb['player_state'] == 'INTL').sum()}")

# Season coverage
print(f"\nDataset Coverage:")
print(f"  Seasons: {swac_fb['season'].min()} - {swac_fb['season'].max()}")
print(f"  Years of data: {swac_fb['season'].max() - swac_fb['season'].min() + 1}")

print(f"\n✅ Dataset ready for recruitment analysis!")
print(f"🎯 Key columns: 'player_state', 'team_state', 'is_international'")

# Display final dataset structure
print(f"\nFinal dataset preview:")
swac_fb.head()

=== SWAC FOOTBALL RECRUITMENT DATASET SUMMARY ===

Teams and their home states:
  Alabama A&M (AL): 1,203 players
  Alabama State (AL): 1,588 players
  Alcorn State (MS): 1,295 players
  Bethune-Cookman (FL): 1,455 players
  Florida A&M (FL): 1,508 players
  Grambling (LA): 1,387 players
  Jackson State (MS): 746 players
  Mississippi Valley State (MS): 1,090 players
  Prairie View A&M (TX): 1,481 players
  Southern (LA): 1,313 players
  Texas Southern (TX): 1,334 players
  UAPB (AR): 1,171 players

Top 10 states by player count:
  FL: 3,404 players (21.9%)
  TX: 2,648 players (17.0%)
  LA: 2,120 players (13.6%)
  AL: 1,528 players (9.8%)
  MS: 1,414 players (9.1%)
  GA: 1,212 players (7.8%)
  CA: 468 players (3.0%)
  TN: 382 players (2.5%)
  AR: 378 players (2.4%)
  None: 268 players (1.7%)

Data Quality Metrics:
  Total records: 15,571
  Complete player_state: 98.28%
  Complete team_state: 100.00%
  International players: 87

Dataset Coverage:
  Seasons: 2010 - 2025
  Years of data: 

,team,season,name,high_school,hometown,previous_school,class,team_state,player_state,is_international
0,Jackson State,2025,Travis Terrell Jr.,Creekside HS,"Atlanta, Ga.",None,So.,MS,GA,False
1,Jackson State,2025,Jeremiah Williams,Holmes County Central HS,"Lexington, Miss.",None,R-Sr.,MS,MS,False
2,Jackson State,2025,Khamauri Rogers,Holmes County Central HS,"Madison, Miss.",Mississippi State,Gr.,MS,MS,False
3,Jackson State,2025,Shemar Savage,Lompoc HS,"Lompoc, Calif.",Prairie View A&M,Gr.,MS,CA,False
4,Jackson State,2025,Ja'Naylon Dupree,Neshoba Central HS,"Philadelphia, Miss.",Mississippi Gulf Coast CC,Sr.,MS,MS,False


In [17]:
# ===== DETAILED ANALYSIS OF MISSING PLAYERS =====

print("=== PLAYERS STILL MISSING STATE ASSIGNMENTS ===\n")

# Get all players still missing state assignments
missing_players = swac_fb[swac_fb['player_state'].isnull()].copy()

print(f"Total missing players: {len(missing_players)}")
print(f"Percentage of total: {len(missing_players)/len(swac_fb)*100:.2f}%\n")

# Breakdown by team
print("Missing players by team:")
team_missing = missing_players['team'].value_counts().sort_values(ascending=False)
for team, count in team_missing.items():
    total_team = len(swac_fb[swac_fb['team'] == team])
    percentage = count/total_team*100
    print(f"  {team}: {count} missing ({percentage:.1f}% of {total_team} players)")

print(f"\n" + "="*60)

# Categorize missing players by available data
print("Breakdown by available location data:")
missing_categories = {
    'No data at all': 0,
    'Hometown only': 0,
    'High school only': 0,
    'Previous school only': 0,
    'Hometown + High school': 0,
    'Hometown + Previous school': 0,
    'High school + Previous school': 0,
    'All fields present': 0
}

for idx, row in missing_players.iterrows():
    has_hometown = pd.notna(row['hometown']) and str(row['hometown']).strip() != ''
    has_hs = pd.notna(row['high_school']) and str(row['high_school']).strip() != ''
    has_prev = pd.notna(row['previous_school']) and str(row['previous_school']).strip() != ''
    
    if has_hometown and has_hs and has_prev:
        missing_categories['All fields present'] += 1
    elif has_hometown and has_hs:
        missing_categories['Hometown + High school'] += 1
    elif has_hometown and has_prev:
        missing_categories['Hometown + Previous school'] += 1
    elif has_hs and has_prev:
        missing_categories['High school + Previous school'] += 1
    elif has_hometown:
        missing_categories['Hometown only'] += 1
    elif has_hs:
        missing_categories['High school only'] += 1
    elif has_prev:
        missing_categories['Previous school only'] += 1
    else:
        missing_categories['No data at all'] += 1

for category, count in missing_categories.items():
    if count > 0:
        print(f"  {category}: {count} players")

print(f"\n" + "="*60)

# Show sample of each category
print("SAMPLE ENTRIES BY CATEGORY:\n")

# Players with all data but still missing state
all_data_missing = missing_players[
    missing_players['hometown'].notna() & 
    missing_players['high_school'].notna() & 
    missing_players['previous_school'].notna()
]
if len(all_data_missing) > 0:
    print(f"Players with ALL data but no state extracted ({len(all_data_missing)}):")
    print(all_data_missing[['name', 'team', 'hometown', 'high_school']].head(10))
    print()

# Players with hometown only
hometown_only = missing_players[
    missing_players['hometown'].notna() & 
    missing_players['high_school'].isnull() & 
    missing_players['previous_school'].isnull()
]
if len(hometown_only) > 0:
    print(f"Players with hometown only ({len(hometown_only)}):")
    print(hometown_only[['name', 'team', 'hometown']].head(10))
    print()

# Players with no data at all
no_data = missing_players[
    missing_players['hometown'].isnull() & 
    missing_players['high_school'].isnull() & 
    missing_players['previous_school'].isnull()
]
if len(no_data) > 0:
    print(f"Players with NO location data ({len(no_data)}):")
    print(no_data[['name', 'team', 'season']].head(10))
    print()

# Show the full missing dataset for inspection
print("="*60)
print("COMPLETE MISSING PLAYERS DATASET:")
print("="*60)
missing_players[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']]

=== PLAYERS STILL MISSING STATE ASSIGNMENTS ===

Total missing players: 268
Percentage of total: 1.72%

Missing players by team:
  Bethune-Cookman: 77 missing (5.3% of 1455 players)
  Alabama A&M: 29 missing (2.4% of 1203 players)
  Florida A&M: 28 missing (1.9% of 1508 players)
  Alcorn State: 23 missing (1.8% of 1295 players)
  Southern: 21 missing (1.6% of 1313 players)
  Alabama State: 19 missing (1.2% of 1588 players)
  Prairie View A&M: 19 missing (1.3% of 1481 players)
  Texas Southern: 18 missing (1.3% of 1334 players)
  Grambling: 11 missing (0.8% of 1387 players)
  UAPB: 11 missing (0.9% of 1171 players)
  Mississippi Valley State: 7 missing (0.6% of 1090 players)
  Jackson State: 5 missing (0.7% of 746 players)

Breakdown by available location data:
  No data at all: 72 players
  Hometown only: 37 players
  High school only: 1 players
  Previous school only: 1 players
  Hometown + High school: 93 players
  Hometown + Previous school: 45 players
  All fields present: 19 playe

,name,team,season,hometown,high_school,previous_school
335,Delano Salgado,Jackson State,2022,"Laveen Village, Az.",Mountain Pointe HS,Georgetown
606,Alexander Shaw,Jackson State,2019,"Vicksburg, Miss,",NaN,Warren Central HS
660,Jarrad Hayes,Jackson State,2018,"Central, Lousiana",NaN,None
681,Juavon Brown,Jackson State,2018,"Mt. Belvieu, Lousiana",NaN,None
688,Alexander Shaw,Jackson State,2018,"Vicksburg, Miss,",NaN,None
...,...,...,...,...,...,...
15481,John Powers,Bethune-Cookman,2011,",",NaN,None
15482,LeBrandon Richardson,Bethune-Cookman,2011,",",NaN,None
15483,Terry Williams,Bethune-Cookman,2011,",",NaN,None
15533,Arlen McCray - Nibbs,Bethune-Cookman,2010,"Atlanta, Ga .",North Cobb HS,None


In [18]:
# ===== TARGETED CITY FIXES =====

print("Applying targeted city-to-state fixes...")
corrections_applied = 0

# Define the specific city mappings
city_mappings = {
    'Benton Harbor': 'MI',
    'Cibolo': 'TX',
    'Deerfield Beach': 'FL',
    'DentonTx': 'TX',
    'Duncanville': 'TX',
    'The Woodlands': 'TX',
    'Ft. Lauderdale': 'FL',
    'Grand Rapids': 'MI',
    'Honolulu': 'HI',
    'Killeen': 'TX',
    'Natch': 'MS',
    'Olosega': 'INTL',
    'Port Arthur': 'TX',
    'Port St. Lucie': 'FL',
    'Sarasota': 'FL',
    'Seattle': 'WA',
    'Tallahassee': 'FL'
}

# Apply each city mapping
for city, state in city_mappings.items():
    # Look for exact city name in hometown (case insensitive)
    city_mask = (swac_fb['hometown'].str.contains(city, na=False, case=False)) & \
                (swac_fb['player_state'].isnull())
    
    if city_mask.any():
        swac_fb.loc[city_mask, 'player_state'] = state
        corrections_applied += city_mask.sum()
        print(f"✓ Fixed {city} → {state}: {city_mask.sum()} corrections")

# Calculate new statistics
final_missing_after_city_fixes = swac_fb['player_state'].isnull().sum()
final_success_rate_new = ((len(swac_fb) - final_missing_after_city_fixes) / len(swac_fb) * 100)

print(f"\n🎯 CITY CORRECTIONS SUMMARY:")
print(f"Total corrections applied: {corrections_applied}")
print(f"Missing count before city fixes: 268")  
print(f"Missing count after city fixes: {final_missing_after_city_fixes}")
print(f"Improvement: {268 - final_missing_after_city_fixes} fewer missing")
print(f"New success rate: {final_success_rate_new:.2f}%")

print(f"\nRemaining missing players: {final_missing_after_city_fixes}")

Applying targeted city-to-state fixes...
✓ Fixed Benton Harbor → MI: 5 corrections
✓ Fixed Cibolo → TX: 1 corrections
✓ Fixed Deerfield Beach → FL: 1 corrections
✓ Fixed DentonTx → TX: 5 corrections
✓ Fixed Duncanville → TX: 2 corrections
✓ Fixed Ft. Lauderdale → FL: 2 corrections
✓ Fixed Grand Rapids → MI: 1 corrections
✓ Fixed Honolulu → HI: 4 corrections
✓ Fixed Killeen → TX: 4 corrections
✓ Fixed Natch → MS: 1 corrections
✓ Fixed Olosega → INTL: 6 corrections
✓ Fixed Port Arthur → TX: 3 corrections
✓ Fixed Port St. Lucie → FL: 6 corrections
✓ Fixed Sarasota → FL: 9 corrections
✓ Fixed Seattle → WA: 2 corrections
✓ Fixed Tallahassee → FL: 2 corrections

🎯 CITY CORRECTIONS SUMMARY:
Total corrections applied: 54
Missing count before city fixes: 268
Missing count after city fixes: 214
Improvement: 54 fewer missing
New success rate: 98.63%

Remaining missing players: 214


In [ ]:
final_missing_after_city_fixes 

In [19]:
# ===== UPDATED MISSING PLAYERS AFTER CITY FIXES =====

print("=== MISSING PLAYERS AFTER CITY FIXES ===\n")

# Create updated missing players dataframe
missing_players_updated = swac_fb[swac_fb['player_state'].isnull()].copy()

print(f"Total missing players: {len(missing_players_updated)}")
print(f"Percentage of total: {len(missing_players_updated)/len(swac_fb)*100:.2f}%\n")

# Breakdown by team
print("Missing players by team:")
team_missing_updated = missing_players_updated['team'].value_counts().sort_values(ascending=False)
for team, count in team_missing_updated.items():
    total_team = len(swac_fb[swac_fb['team'] == team])
    percentage = count/total_team*100
    print(f"  {team}: {count} missing ({percentage:.1f}% of {total_team} players)")

print(f"\n" + "="*80)
print("SAMPLE OF REMAINING MISSING ENTRIES:")
print("="*80)

# Show a sample of what's still missing with all relevant columns
sample_size = min(20, len(missing_players_updated))
sample_display = missing_players_updated[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].head(sample_size)
print(sample_display.to_string(index=False))

print(f"\n" + "="*80)
print("BREAKDOWN BY DATA AVAILABILITY:")
print("="*80)

# Categorize the remaining missing players
categories = {
    'Empty/null hometown, has high school': 0,
    'Empty/null hometown, no high school': 0,
    'Hometown = comma only': 0,
    'Hometown has data but no state': 0,
    'International patterns': 0,
    'Completely empty': 0
}

intl_indicators = ['Australia', 'Canada', 'Germany', 'Nigeria', 'Virgin Islands', 'V.I.', 'American Samoa', 'Jamaica', 'Bahamas']

for idx, row in missing_players_updated.iterrows():
    hometown = str(row['hometown']) if pd.notna(row['hometown']) else ''
    high_school = str(row['high_school']) if pd.notna(row['high_school']) else ''
    
    # Check for international patterns
    is_intl = any(indicator.lower() in hometown.lower() for indicator in intl_indicators)
    
    if is_intl:
        categories['International patterns'] += 1
    elif hometown.strip() == '' or hometown == 'nan':
        if high_school.strip() != '' and high_school != 'nan':
            categories['Empty/null hometown, has high school'] += 1
        else:
            categories['Empty/null hometown, no high school'] += 1
    elif hometown.strip() == ',':
        categories['Hometown = comma only'] += 1
    elif len(hometown.strip()) == 0 and len(high_school.strip()) == 0:
        categories['Completely empty'] += 1
    else:
        categories['Hometown has data but no state'] += 1

for category, count in categories.items():
    if count > 0:
        print(f"  {category}: {count} players")

print(f"\n" + "="*80)
print("FULL DATASET FOR ANALYSIS:")
print("="*80)

# Display the complete remaining missing dataset
missing_players_updated[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']]

=== MISSING PLAYERS AFTER CITY FIXES ===

Total missing players: 214
Percentage of total: 1.37%

Missing players by team:
  Bethune-Cookman: 58 missing (4.0% of 1455 players)
  Alabama A&M: 27 missing (2.2% of 1203 players)
  Florida A&M: 26 missing (1.7% of 1508 players)
  Southern: 21 missing (1.6% of 1313 players)
  Alabama State: 19 missing (1.2% of 1588 players)
  Alcorn State: 16 missing (1.2% of 1295 players)
  Grambling: 11 missing (0.8% of 1387 players)
  UAPB: 10 missing (0.9% of 1171 players)
  Prairie View A&M: 9 missing (0.6% of 1481 players)
  Mississippi Valley State: 7 missing (0.6% of 1090 players)
  Jackson State: 5 missing (0.7% of 746 players)
  Texas Southern: 5 missing (0.4% of 1334 players)

SAMPLE OF REMAINING MISSING ENTRIES:
            name          team  season              hometown        high_school      previous_school
  Delano Salgado Jackson State    2022   Laveen Village, Az. Mountain Pointe HS           Georgetown
  Alexander Shaw Jackson State    201

,name,team,season,hometown,high_school,previous_school
335,Delano Salgado,Jackson State,2022,"Laveen Village, Az.",Mountain Pointe HS,Georgetown
606,Alexander Shaw,Jackson State,2019,"Vicksburg, Miss,",NaN,Warren Central HS
660,Jarrad Hayes,Jackson State,2018,"Central, Lousiana",NaN,None
681,Juavon Brown,Jackson State,2018,"Mt. Belvieu, Lousiana",NaN,None
688,Alexander Shaw,Jackson State,2018,"Vicksburg, Miss,",NaN,None
...,...,...,...,...,...,...
15481,John Powers,Bethune-Cookman,2011,",",NaN,None
15482,LeBrandon Richardson,Bethune-Cookman,2011,",",NaN,None
15483,Terry Williams,Bethune-Cookman,2011,",",NaN,None
15533,Arlen McCray - Nibbs,Bethune-Cookman,2010,"Atlanta, Ga .",North Cobb HS,None


In [20]:
# ===== SPECIFIC PLAYER CORRECTIONS =====

print("Applying specific player corrections...")

# Fix Andre Washington from Alcorn State
andre_mask = (swac_fb['name'] == 'Andre Washington') & \
             (swac_fb['team'] == 'Alcorn State') & \
             (swac_fb['high_school'].str.contains('Ridgeview HS/UNC Charlotte', na=False))

if andre_mask.any():
    print("Found Andre Washington - fixing his data...")
    swac_fb.loc[andre_mask, 'high_school'] = 'Ridgeview HS'
    swac_fb.loc[andre_mask, 'previous_school'] = 'UNC Charlotte'
    swac_fb.loc[andre_mask, 'hometown'] = 'Hopkins, SC'
    swac_fb.loc[andre_mask, 'player_state'] = 'SC'
    print("✓ Andre Washington corrected: Hopkins, SC → SC")
else:
    print("Andre Washington not found with expected pattern")

# Check current missing count after Andre Washington fix
current_missing = swac_fb['player_state'].isnull().sum()
current_success_rate = ((len(swac_fb) - current_missing) / len(swac_fb) * 100)

print(f"\nAfter Andre Washington fix:")
print(f"Missing players: {current_missing}")
print(f"Success rate: {current_success_rate:.2f}%")

# Show if Andre Washington is now properly assigned
andre_check = swac_fb[(swac_fb['name'] == 'Andre Washington') & (swac_fb['team'] == 'Alcorn State')]
if len(andre_check) > 0:
    print(f"\nAndre Washington status check:")
    print(andre_check[['name', 'team', 'hometown', 'high_school', 'previous_school', 'player_state']].to_string(index=False))

Applying specific player corrections...
Found Andre Washington - fixing his data...
✓ Andre Washington corrected: Hopkins, SC → SC

After Andre Washington fix:
Missing players: 213
Success rate: 98.63%

Andre Washington status check:
            name         team    hometown  high_school previous_school player_state
Andre Washington Alcorn State Hopkins, SC Ridgeview HS   UNC Charlotte           SC


In [21]:
# ===== COMPREHENSIVE INDIVIDUAL IRREGULARITY FIXES =====

print("Applying comprehensive individual irregularity fixes...")
total_corrections = 0

# 1. Waterloo, Ia. → IA
mask = swac_fb['hometown'] == 'Waterloo, Ia.'
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'IA'
    total_corrections += mask.sum()
    print(f"✓ Fixed Waterloo, Ia. → IA: {mask.sum()} corrections")

# 2. Alexander Shaw for Jackson State → Vicksburg, MS
mask = (swac_fb['name'] == 'Alexander Shaw') & (swac_fb['team'] == 'Jackson State')
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Vicksburg, MS'
    swac_fb.loc[mask, 'player_state'] = 'MS'
    total_corrections += mask.sum()
    print(f"✓ Fixed Alexander Shaw → Vicksburg, MS: {mask.sum()} corrections")

# 3. Marquell Rozier for Bethune-Cookman → NC
mask = (swac_fb['name'] == 'Marquell Rozier') & (swac_fb['team'] == 'Bethune-Cookman')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'NC'
    total_corrections += mask.sum()
    print(f"✓ Fixed Marquell Rozier → NC: {mask.sum()} corrections")

# 4. Tucker, Ga, → Tucker, GA
mask = swac_fb['hometown'] == 'Tucker, Ga,'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Tucker, GA'
    swac_fb.loc[mask, 'player_state'] = 'GA'
    total_corrections += mask.sum()
    print(f"✓ Fixed Tucker, Ga, → Tucker, GA: {mask.sum()} corrections")

# 5. Hometowns ending in "Aus." or "AU" → INTL
mask = swac_fb['hometown'].str.endswith(('Aus.', 'AU'), na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'INTL'
    total_corrections += mask.sum()
    print(f"✓ Fixed Aus./AU endings → INTL: {mask.sum()} corrections")

# 6. Hometowns ending in "Misourri" → MO
mask = swac_fb['hometown'].str.endswith('Misourri', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'MO'
    total_corrections += mask.sum()
    print(f"✓ Fixed Misourri ending → MO: {mask.sum()} corrections")

# 7. Tekeven Thomas for Bethune-Cookman → AL
mask = (swac_fb['name'] == 'Tekeven Thomas') & (swac_fb['team'] == 'Bethune-Cookman')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'AL'
    total_corrections += mask.sum()
    print(f"✓ Fixed Tekeven Thomas → AL: {mask.sum()} corrections")

# 8. Warner Robins → Warner Robins, GA
mask = swac_fb['hometown'] == 'Warner Robins'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Warner Robins, GA'
    swac_fb.loc[mask, 'player_state'] = 'GA'
    total_corrections += mask.sum()
    print(f"✓ Fixed Warner Robins → Warner Robins, GA: {mask.sum()} corrections")

# 9. Hometowns ending in "Ga," → GA
mask = swac_fb['hometown'].str.endswith('Ga,', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'GA'
    total_corrections += mask.sum()
    print(f"✓ Fixed Ga, endings → GA: {mask.sum()} corrections")

# 10. Brandon Duncan for UAPB → NY
mask = (swac_fb['name'] == 'Brandon Duncan') & (swac_fb['team'] == 'UAPB')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'NY'
    total_corrections += mask.sum()
    print(f"✓ Fixed Brandon Duncan → NY: {mask.sum()} corrections")

# 11. Austin Jones from Alabama A&M → IL
mask = (swac_fb['name'] == 'Austin Jones') & (swac_fb['team'] == 'Alabama A&M')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'IL'
    total_corrections += mask.sum()
    print(f"✓ Fixed Austin Jones → IL: {mask.sum()} corrections")

# 12. Hometowns ending in "SK" → INTL
mask = swac_fb['hometown'].str.endswith('SK', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'INTL'
    total_corrections += mask.sum()
    print(f"✓ Fixed SK endings → INTL: {mask.sum()} corrections")

# 13. Hometowns ending in "Lo." → LA
mask = swac_fb['hometown'].str.endswith('Lo.', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'LA'
    total_corrections += mask.sum()
    print(f"✓ Fixed Lo. endings → LA: {mask.sum()} corrections")

# 14. Hometowns ending in "Missississippi" or "Mississisippi" → MS
mask = swac_fb['hometown'].str.endswith(('Missississippi', 'Mississisippi'), na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'MS'
    total_corrections += mask.sum()
    print(f"✓ Fixed Mississippi misspellings → MS: {mask.sum()} corrections")

# 15. Hometowns ending in "Ont." or "Ont" → INTL
mask = swac_fb['hometown'].str.endswith(('Ont.', 'Ont'), na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'INTL'
    total_corrections += mask.sum()
    print(f"✓ Fixed Ont./Ont. endings → INTL: {mask.sum()} corrections")

# 16. Eric Smith for Florida A&M → Opa Locka, FL
mask = (swac_fb['name'] == 'Eric Smith') & (swac_fb['team'] == 'Florida A&M')
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Opa Locka, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    swac_fb.loc[mask, 'high_school'] = 'Norland HS'
    total_corrections += mask.sum()
    print(f"✓ Fixed Eric Smith (FAMU) → Opa Locka, FL: {mask.sum()} corrections")

# 17. Keir Abrams for Florida A&M → Oakland, CA
mask = (swac_fb['name'] == 'Keir Abrams') & (swac_fb['team'] == 'Florida A&M')
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Oakland, CA'
    swac_fb.loc[mask, 'player_state'] = 'CA'
    total_corrections += mask.sum()
    print(f"✓ Fixed Keir Abrams → Oakland, CA: {mask.sum()} corrections")

# 18. Juavon Brown for Jackson State → LA
mask = (swac_fb['name'] == 'Juavon Brown') & (swac_fb['team'] == 'Jackson State')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'LA'
    total_corrections += mask.sum()
    print(f"✓ Fixed Juavon Brown → LA: {mask.sum()} corrections")

# 19. Hometowns ending in "A.S." → INTL
mask = swac_fb['hometown'].str.endswith('A.S.', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'INTL'
    total_corrections += mask.sum()
    print(f"✓ Fixed A.S. endings → INTL: {mask.sum()} corrections")

# 20. Hometowns ending in "Fla,." → FL
mask = swac_fb['hometown'].str.endswith('Fla,.', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'FL'
    total_corrections += mask.sum()
    print(f"✓ Fixed Fla,. endings → FL: {mask.sum()} corrections")

# 21. Hometowns ending in "Mi" or "Mi." → MI
mask = swac_fb['hometown'].str.endswith(('Mi', 'Mi.'), na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'MI'
    total_corrections += mask.sum()
    print(f"✓ Fixed Mi/Mi. endings → MI: {mask.sum()} corrections")

# 22. Hometowns ending in "Flo" → FL
mask = swac_fb['hometown'].str.endswith('Flo', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'FL'
    total_corrections += mask.sum()
    print(f"✓ Fixed Flo endings → FL: {mask.sum()} corrections")

# 23. Devin Tribble for Alcorn State → MS
mask = (swac_fb['name'] == 'Devin Tribble') & (swac_fb['team'] == 'Alcorn State')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'MS'
    total_corrections += mask.sum()
    print(f"✓ Fixed Devin Tribble → MS: {mask.sum()} corrections")

# 24. Hometowns ending in "Ak." → AK
mask = swac_fb['hometown'].str.endswith('Ak.', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'AK'
    total_corrections += mask.sum()
    print(f"✓ Fixed Ak. endings → AK: {mask.sum()} corrections")

# 25. Hometowns ending in "Geo." → GA
mask = swac_fb['hometown'].str.endswith('Geo.', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'GA'
    total_corrections += mask.sum()
    print(f"✓ Fixed Geo. endings → GA: {mask.sum()} corrections")

# 26. Antonio Wells for Alcorn State → MS
mask = (swac_fb['name'] == 'Antonio Wells') & (swac_fb['team'] == 'Alcorn State')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'MS'
    total_corrections += mask.sum()
    print(f"✓ Fixed Antonio Wells → MS: {mask.sum()} corrections")

# 27. Keshuan Blackmon for Bethune-Cookman → MS
mask = (swac_fb['name'] == 'Keshuan Blackmon') & (swac_fb['team'] == 'Bethune-Cookman')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'MS'
    total_corrections += mask.sum()
    print(f"✓ Fixed Keshuan Blackmon → MS: {mask.sum()} corrections")

# 28. Jamie Gleaton for Texas Southern → Batesburg-Leesville, SC
mask = (swac_fb['name'] == 'Jamie Gleaton') & (swac_fb['team'] == 'Texas Southern')
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Batesburg-Leesville, SC'
    swac_fb.loc[mask, 'high_school'] = 'Batesburg-Leesville HS'
    swac_fb.loc[mask, 'player_state'] = 'SC'
    total_corrections += mask.sum()
    print(f"✓ Fixed Jamie Gleaton → Batesburg-Leesville, SC: {mask.sum()} corrections")

# 29. Hometowns ending in ".Ga.", "Ga .", "Ga,", "Geo." → GA
mask = swac_fb['hometown'].str.endswith(('.Ga.', 'Ga .', 'Ga,', 'Geo.'), na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'GA'
    total_corrections += mask.sum()
    print(f"✓ Fixed various GA endings → GA: {mask.sum()} corrections")

# Calculate final statistics after all corrections
final_missing_after_irregularities = swac_fb['player_state'].isnull().sum()
final_success_rate_after_irregularities = ((len(swac_fb) - final_missing_after_irregularities) / len(swac_fb) * 100)

print(f"\n🎯 IRREGULARITY CORRECTIONS SUMMARY:")
print(f"Total corrections applied: {total_corrections}")
print(f"Missing count after irregularity fixes: {final_missing_after_irregularities}")
print(f"New success rate: {final_success_rate_after_irregularities:.2f}%")
print(f"Improvement from previous: {final_missing_after_city_fixes - final_missing_after_irregularities} fewer missing")

Applying comprehensive individual irregularity fixes...
✓ Fixed Waterloo, Ia. → IA: 1 corrections
✓ Fixed Alexander Shaw → Vicksburg, MS: 3 corrections
✓ Fixed Marquell Rozier → NC: 3 corrections
✓ Fixed Tucker, Ga, → Tucker, GA: 3 corrections
✓ Fixed Aus./AU endings → INTL: 3 corrections
✓ Fixed Misourri ending → MO: 2 corrections
✓ Fixed Tekeven Thomas → AL: 1 corrections
✓ Fixed Warner Robins → Warner Robins, GA: 1 corrections
✓ Fixed Ga, endings → GA: 5 corrections
✓ Fixed Brandon Duncan → NY: 4 corrections
✓ Fixed Austin Jones → IL: 4 corrections
✓ Fixed SK endings → INTL: 2 corrections
✓ Fixed Lo. endings → LA: 3 corrections
✓ Fixed Mississippi misspellings → MS: 3 corrections
✓ Fixed Ont./Ont. endings → INTL: 6 corrections
✓ Fixed Eric Smith (FAMU) → Opa Locka, FL: 6 corrections
✓ Fixed Keir Abrams → Oakland, CA: 1 corrections
✓ Fixed Juavon Brown → LA: 1 corrections
✓ Fixed A.S. endings → INTL: 8 corrections
✓ Fixed Fla,. endings → FL: 3 corrections
✓ Fixed Mi/Mi. endings → MI:

In [22]:
# ===== NEW MISSING PLAYERS DATAFRAME AFTER ALL FIXES =====

print("=== CREATING NEW MISSING PLAYERS DATAFRAME ===\n")

# Create the final missing players dataframe after all corrections
final_missing_players = swac_fb[swac_fb['player_state'].isnull()].copy()

print(f"Remaining missing players: {len(final_missing_players)}")
print(f"Percentage of total: {len(final_missing_players)/len(swac_fb)*100:.2f}%\n")

# Summary by team
print("Remaining missing by team:")
team_missing_final = final_missing_players['team'].value_counts().sort_values(ascending=False)
for team, count in team_missing_final.items():
    total_team = len(swac_fb[swac_fb['team'] == team])
    percentage = count/total_team*100
    print(f"  {team}: {count} missing ({percentage:.1f}% of {total_team} players)")

print(f"\n" + "="*80)
print("NEXT GROUP TO ANALYZE:")
print("="*80)

# Show the next batch of missing players for analysis
print(final_missing_players[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].head(30).to_string(index=False))

print(f"\n" + "="*80)
print("COMPLETE REMAINING MISSING DATASET:")
print("="*80)

# Display the complete final missing dataset for further analysis
final_missing_players[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']]

=== CREATING NEW MISSING PLAYERS DATAFRAME ===

Remaining missing players: 139
Percentage of total: 0.89%

Remaining missing by team:
  Bethune-Cookman: 44 missing (3.0% of 1455 players)
  Alabama A&M: 24 missing (2.0% of 1203 players)
  Southern: 19 missing (1.4% of 1313 players)
  Grambling: 11 missing (0.8% of 1387 players)
  Alabama State: 10 missing (0.6% of 1588 players)
  Florida A&M: 8 missing (0.5% of 1508 players)
  Alcorn State: 8 missing (0.6% of 1295 players)
  Prairie View A&M: 7 missing (0.5% of 1481 players)
  UAPB: 4 missing (0.3% of 1171 players)
  Jackson State: 2 missing (0.3% of 746 players)
  Texas Southern: 2 missing (0.1% of 1334 players)

NEXT GROUP TO ANALYZE:
             name          team  season            hometown        high_school      previous_school
   Delano Salgado Jackson State    2022 Laveen Village, Az. Mountain Pointe HS           Georgetown
     Jarrad Hayes Jackson State    2018   Central, Lousiana                NaN                 None
    J

,name,team,season,hometown,high_school,previous_school
335,Delano Salgado,Jackson State,2022,"Laveen Village, Az.",Mountain Pointe HS,Georgetown
660,Jarrad Hayes,Jackson State,2018,"Central, Lousiana",NaN,None
1189,Jeremy Jarman,Alabama State,2022,"Cantonment, Fla,",NaN,Escambia High School
1753,Darrel King,Alabama State,2016,NaN,NaN,None
1757,Manny Holmes,Alabama State,2016,NaN,NaN,None
...,...,...,...,...,...,...
15478,Jeff Fagan,Bethune-Cookman,2011,",",NaN,None
15480,Darian McCaskill,Bethune-Cookman,2011,",",NaN,None
15481,John Powers,Bethune-Cookman,2011,",",NaN,None
15482,LeBrandon Richardson,Bethune-Cookman,2011,",",NaN,None


In [25]:
# ===== ROUND 2: ADDITIONAL IRREGULAR CORRECTIONS =====

print("Applying Round 2 of irregular corrections...")
round2_corrections = 0

# 1. Hometown ending in "Wi." → WI
mask = swac_fb['hometown'].str.endswith('Wi.', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'WI'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Wi. endings → WI: {mask.sum()} corrections")

# 2. Hometown ending in "Az." → AZ
mask = swac_fb['hometown'].str.endswith('Az.', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'AZ'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Az. endings → AZ: {mask.sum()} corrections")

# 3. Bennie Peoples for Grambling State (2010 and 2011)
mask = ((swac_fb['name'] == 'Bennie Peoples') | (swac_fb['name'] == 'Bennie, Peoples,')) & \
       (swac_fb['team'] == 'Grambling') & \
       (swac_fb['season'].isin([2010, 2011]))
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Vicksburg, MS'
    swac_fb.loc[mask, 'player_state'] = 'MS'
    swac_fb.loc[mask, 'previous_school'] = 'Co-Lin CC'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Bennie Peoples → Vicksburg, MS: {mask.sum()} corrections")

# 4. Radonte Womack for Bethune-Cookman in 2021 → MS
mask = (swac_fb['name'] == 'Radonte Womack') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2021)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'MS'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Radonte Womack → MS: {mask.sum()} corrections")

# 5. DeAngelo Alexander of Prairie View A&M in 2022 → TX
mask = (swac_fb['name'] == 'DeAngelo Alexander') & \
       (swac_fb['team'] == 'Prairie View A&M') & \
       (swac_fb['season'] == 2022)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'TX'
    round2_corrections += mask.sum()
    print(f"✓ Fixed DeAngelo Alexander → TX: {mask.sum()} corrections")

# 6. Hometown ending in "Louisanna" or "Lousiana" → LA
mask = swac_fb['hometown'].str.endswith(('Louisanna', 'Lousiana'), na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'LA'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Louisiana misspellings → LA: {mask.sum()} corrections")

# 7. Hometown ending in "Forida" → FL
mask = swac_fb['hometown'].str.endswith('Forida', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Forida → FL: {mask.sum()} corrections")

# 8. Hollywood, Fra. → Hollywood, FL
mask = swac_fb['hometown'] == 'Hollywood, Fra.'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Hollywood, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Hollywood, Fra. → Hollywood, FL: {mask.sum()} corrections")

# 9. Grambling State 2011 with previous school "Hodge HS)"
mask = (swac_fb['team'] == 'Grambling') & \
       (swac_fb['season'] == 2011) & \
       (swac_fb['previous_school'].str.contains('Hodge HS)', na=False, regex=False))
if mask.any():
    swac_fb.loc[mask, 'high_school'] = 'Jonesboro Hodge HS'
    swac_fb.loc[mask, 'hometown'] = 'Jonesboro, LA'
    swac_fb.loc[mask, 'player_state'] = 'LA'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Grambling 2011 Hodge HS players → Jonesboro, LA: {mask.sum()} corrections")

# 10. Van, Phillips, for Grambling State in 2011
mask = (swac_fb['name'] == 'Van, Phillips,') & \
       (swac_fb['team'] == 'Grambling') & \
       (swac_fb['season'] == 2011)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Irondale, AL'
    swac_fb.loc[mask, 'player_state'] = 'AL'
    swac_fb.loc[mask, 'high_school'] = 'Shades Valley HS'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Van, Phillips, → Irondale, AL: {mask.sum()} corrections")

# 11. Names starting with "Stephen" for Grambling State in 2010/2011
mask = (swac_fb['name'].str.startswith('Stephen', na=False)) & \
       (swac_fb['team'] == 'Grambling') & \
       (swac_fb['season'].isin([2010, 2011]))
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Eight Mile, AL'
    swac_fb.loc[mask, 'high_school'] = 'McGill-Toolen HS'
    swac_fb.loc[mask, 'player_state'] = 'AL'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Stephen names → Eight Mile, AL: {mask.sum()} corrections")

# 12. Hometown ending in "Fla," → FL
mask = swac_fb['hometown'].str.endswith('Fla,', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Fla, endings → FL: {mask.sum()} corrections")

# 13. Marc Lucien for UAPB in 2015 → NY
mask = (swac_fb['name'] == 'Marc Lucien') & \
       (swac_fb['team'] == 'UAPB') & \
       (swac_fb['season'] == 2015)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'NY'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Marc Lucien → NY: {mask.sum()} corrections")

# 14. Reggie Polite for Bethune-Cookman in 2013, 2014, 2015 → Bartow, FL
mask = (swac_fb['name'] == 'Reggie Polite') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'].isin([2013, 2014, 2015]))
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Bartow, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Reggie Polite → Bartow, FL: {mask.sum()} corrections")

# 15. Ja'sion Greathouse for Southern in 2022 → IL
mask = (swac_fb['name'] == "Ja'sion Greathouse") & \
       (swac_fb['team'] == 'Southern') & \
       (swac_fb['season'] == 2022)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'IL'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Ja'sion Greathouse → IL: {mask.sum()} corrections")

# 16. Raequan Prince from UAPB in 2021 → OH
mask = (swac_fb['name'] == 'Raequan Prince') & \
       (swac_fb['team'] == 'UAPB') & \
       (swac_fb['season'] == 2021)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'OH'
    round2_corrections += mask.sum()
    print(f"✓ Fixed Raequan Prince → OH: {mask.sum()} corrections")

# 17. Hometown containing ", Ms" → MS
mask = swac_fb['hometown'].str.contains(', Ms', na=False)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'MS'
    round2_corrections += mask.sum()
    print(f"✓ Fixed hometowns with ', Ms' → MS: {mask.sum()} corrections")

# Calculate statistics after Round 2
missing_after_round2 = swac_fb['player_state'].isnull().sum()
success_rate_after_round2 = ((len(swac_fb) - missing_after_round2) / len(swac_fb) * 100)

print(f"\n🎯 ROUND 2 CORRECTIONS SUMMARY:")
print(f"Round 2 corrections applied: {round2_corrections}")
print(f"Missing count after Round 2: {missing_after_round2}")
print(f"New success rate: {success_rate_after_round2:.2f}%")
print(f"Improvement from Round 1: {final_missing_after_irregularities - missing_after_round2} fewer missing")

# Create updated missing players dataframe
print(f"\n=== UPDATED MISSING PLAYERS AFTER ROUND 2 ===")
missing_after_round2_df = swac_fb[swac_fb['player_state'].isnull()].copy()
print(f"Remaining missing players: {len(missing_after_round2_df)}")

if len(missing_after_round2_df) > 0:
    print(f"\nNext batch for analysis:")
    print(missing_after_round2_df[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].head(20).to_string(index=False))

Applying Round 2 of irregular corrections...
✓ Fixed Wi. endings → WI: 3 corrections
✓ Fixed Az. endings → AZ: 1 corrections
✓ Fixed Bennie Peoples → Vicksburg, MS: 2 corrections
✓ Fixed Radonte Womack → MS: 1 corrections
✓ Fixed DeAngelo Alexander → TX: 1 corrections
✓ Fixed Louisiana misspellings → LA: 3 corrections
✓ Fixed Forida → FL: 1 corrections
✓ Fixed Grambling 2011 Hodge HS players → Jonesboro, LA: 2 corrections
✓ Fixed Van, Phillips, → Irondale, AL: 1 corrections
✓ Fixed Stephen names → Eight Mile, AL: 3 corrections
✓ Fixed Fla, endings → FL: 2 corrections
✓ Fixed Marc Lucien → NY: 1 corrections
✓ Fixed Reggie Polite → Bartow, FL: 3 corrections
✓ Fixed Ja'sion Greathouse → IL: 1 corrections
✓ Fixed Raequan Prince → OH: 1 corrections
✓ Fixed hometowns with ', Ms' → MS: 5 corrections

🎯 ROUND 2 CORRECTIONS SUMMARY:
Round 2 corrections applied: 31
Missing count after Round 2: 114
New success rate: 99.27%
Improvement from Round 1: 25 fewer missing

=== UPDATED MISSING PLAYERS AF

In [26]:
# ===== ROUND 3: FINAL SPECIFIC CORRECTIONS =====

print("Applying Round 3 of specific corrections...")
round3_corrections = 0

# 1. Genoa Sartin for Alcorn State in 2014
mask = (swac_fb['name'] == 'Genoa Sartin') & \
       (swac_fb['team'] == 'Alcorn State') & \
       (swac_fb['season'] == 2014)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Brookhaven, MS'
    swac_fb.loc[mask, 'high_school'] = 'Brookhaven HS'
    swac_fb.loc[mask, 'player_state'] = 'MS'
    round3_corrections += mask.sum()
    print(f"✓ Fixed Genoa Sartin → Brookhaven, MS: {mask.sum()} corrections")

# 2. If hometown is "/ Homestead HS"
mask = swac_fb['hometown'] == '/ Homestead HS'
if mask.any():
    swac_fb.loc[mask, 'high_school'] = 'Homestead HS'
    swac_fb.loc[mask, 'hometown'] = 'Homestead, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round3_corrections += mask.sum()
    print(f"✓ Fixed / Homestead HS → Homestead, FL: {mask.sum()} corrections")

# 3. If hometown is "/ Plantation HS"
mask = swac_fb['hometown'] == '/ Plantation HS'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Plantation, FL'
    swac_fb.loc[mask, 'high_school'] = 'Plantation HS'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round3_corrections += mask.sum()
    print(f"✓ Fixed / Plantation HS → Plantation, FL: {mask.sum()} corrections")

# 4. If hometown is "/ P.K. Yonge HS"
mask = swac_fb['hometown'] == '/ P.K. Yonge HS'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Gainesville, FL'
    swac_fb.loc[mask, 'high_school'] = 'P.K. Yonge HS'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round3_corrections += mask.sum()
    print(f"✓ Fixed / P.K. Yonge HS → Gainesville, FL: {mask.sum()} corrections")

# 5. If hometown is "/ Curie HS"
mask = swac_fb['hometown'] == '/ Curie HS'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Chicago, IL'
    swac_fb.loc[mask, 'player_state'] = 'IL'
    swac_fb.loc[mask, 'high_school'] = 'Curie HS'
    round3_corrections += mask.sum()
    print(f"✓ Fixed / Curie HS → Chicago, IL: {mask.sum()} corrections")

# 6. If hometown is "/ Archbishop Curley"
mask = swac_fb['hometown'] == '/ Archbishop Curley'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Miami, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    swac_fb.loc[mask, 'high_school'] = 'Archbishop Curley'
    round3_corrections += mask.sum()
    print(f"✓ Fixed / Archbishop Curley → Miami, FL: {mask.sum()} corrections")

# 7. If hometown is "/ Jefferson HS"
mask = swac_fb['hometown'] == '/ Jefferson HS'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Tampa, FL'
    swac_fb.loc[mask, 'high_school'] = 'Jefferson HS'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round3_corrections += mask.sum()
    print(f"✓ Fixed / Jefferson HS → Tampa, FL: {mask.sum()} corrections")

# 8. If hometown is "/ Norland HS"
mask = swac_fb['hometown'] == '/ Norland HS'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Miami, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    swac_fb.loc[mask, 'high_school'] = 'Norland HS'
    round3_corrections += mask.sum()
    print(f"✓ Fixed / Norland HS → Miami, FL: {mask.sum()} corrections")

# 9. If hometown is "/ Parkway Academy"
mask = swac_fb['hometown'] == '/ Parkway Academy'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Miramar, FL'
    swac_fb.loc[mask, 'high_school'] = 'Parkway Academy'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round3_corrections += mask.sum()
    print(f"✓ Fixed / Parkway Academy → Miramar, FL: {mask.sum()} corrections")

# 10. If hometown is "Lake Highland Prep"
mask = swac_fb['hometown'] == 'Lake Highland Prep'
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Orlando, FL'
    swac_fb.loc[mask, 'high_school'] = 'Lake Highland Prep'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    round3_corrections += mask.sum()
    print(f"✓ Fixed Lake Highland Prep → Orlando, FL: {mask.sum()} corrections")

# 11. Devin Mitchell for UAPB in 2015
mask = (swac_fb['name'] == 'Devin Mitchell') & \
       (swac_fb['team'] == 'UAPB') & \
       (swac_fb['season'] == 2015)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Perris, CA'
    swac_fb.loc[mask, 'player_state'] = 'CA'
    round3_corrections += mask.sum()
    print(f"✓ Fixed Devin Mitchell → Perris, CA: {mask.sum()} corrections")

# 12. Blain Winston of Southern in 2013
mask = (swac_fb['name'] == 'Blain Winston') & \
       (swac_fb['team'] == 'Southern') & \
       (swac_fb['season'] == 2013)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Monroe, LA'
    swac_fb.loc[mask, 'high_school'] = 'Richwood HS'
    swac_fb.loc[mask, 'previous_school'] = 'UL-Lafayette'
    swac_fb.loc[mask, 'player_state'] = 'LA'
    round3_corrections += mask.sum()
    print(f"✓ Fixed Blain Winston → Monroe, LA: {mask.sum()} corrections")

# 13. Jared Mitchell for Bethune-Cookman in 2013
mask = (swac_fb['name'] == 'Jared Mitchell') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2013)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Chatham, VA'
    swac_fb.loc[mask, 'player_state'] = 'VA'
    swac_fb.loc[mask, 'previous_school'] = 'Ole Miss'
    round3_corrections += mask.sum()
    print(f"✓ Fixed Jared Mitchell → Chatham, VA: {mask.sum()} corrections")

# Calculate statistics after Round 3
missing_after_round3 = swac_fb['player_state'].isnull().sum()
success_rate_after_round3 = ((len(swac_fb) - missing_after_round3) / len(swac_fb) * 100)

print(f"\n🎯 ROUND 3 CORRECTIONS SUMMARY:")
print(f"Round 3 corrections applied: {round3_corrections}")
print(f"Missing count after Round 3: {missing_after_round3}")
print(f"New success rate: {success_rate_after_round3:.2f}%")
print(f"Improvement from Round 2: {missing_after_round2 - missing_after_round3} fewer missing")

# Create updated missing players dataframe
print(f"\n=== UPDATED MISSING PLAYERS AFTER ROUND 3 ===")
missing_after_round3_df = swac_fb[swac_fb['player_state'].isnull()].copy()
print(f"Remaining missing players: {len(missing_after_round3_df)}")

if len(missing_after_round3_df) > 0:
    print(f"\nNext batch for analysis:")
    print(missing_after_round3_df[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].head(20).to_string(index=False))
    
    print(f"\n=== PROGRESS SUMMARY ===")
    print(f"Starting success rate: 95.84%")
    print(f"After city fixes: {final_success_rate_new:.2f}%")
    print(f"After Round 1: {final_success_rate_after_irregularities:.2f}%") 
    print(f"After Round 2: {success_rate_after_round2:.2f}%")
    print(f"After Round 3: {success_rate_after_round3:.2f}%")
    print(f"Total improvement: {success_rate_after_round3 - 95.84:.2f} percentage points")
else:
    print("🎉 ALL PLAYERS HAVE BEEN ASSIGNED STATES!")
    print(f"🏆 FINAL SUCCESS RATE: {success_rate_after_round3:.2f}%")

Applying Round 3 of specific corrections...
✓ Fixed Genoa Sartin → Brookhaven, MS: 1 corrections
✓ Fixed / Homestead HS → Homestead, FL: 3 corrections
✓ Fixed / Plantation HS → Plantation, FL: 3 corrections
✓ Fixed / P.K. Yonge HS → Gainesville, FL: 1 corrections
✓ Fixed / Curie HS → Chicago, IL: 1 corrections
✓ Fixed / Archbishop Curley → Miami, FL: 1 corrections
✓ Fixed / Jefferson HS → Tampa, FL: 1 corrections
✓ Fixed / Norland HS → Miami, FL: 1 corrections
✓ Fixed / Parkway Academy → Miramar, FL: 1 corrections
✓ Fixed Devin Mitchell → Perris, CA: 1 corrections
✓ Fixed Blain Winston → Monroe, LA: 1 corrections
✓ Fixed Jared Mitchell → Chatham, VA: 1 corrections

🎯 ROUND 3 CORRECTIONS SUMMARY:
Round 3 corrections applied: 16
Missing count after Round 3: 98
New success rate: 99.37%
Improvement from Round 2: 16 fewer missing

=== UPDATED MISSING PLAYERS AFTER ROUND 3 ===
Remaining missing players: 98

Next batch for analysis:
             name          team  season hometown high_school

In [27]:
# ===== FINAL ROUND ANALYSIS: PATTERN IDENTIFICATION =====

print("🔍 SETTING UP FINAL ROUND ANALYSIS...")
print(f"Remaining missing players: {len(missing_after_round3_df)}")

# Create comprehensive analysis dataframe
final_round_df = missing_after_round3_df.copy()

# Add analysis columns
final_round_df['has_hometown_data'] = final_round_df['hometown'].notna() & (final_round_df['hometown'] != 'None')
final_round_df['has_hs_data'] = final_round_df['high_school'].notna() & (final_round_df['high_school'] != 'None')
final_round_df['has_prev_data'] = final_round_df['previous_school'].notna() & (final_round_df['previous_school'] != 'None')

# Categorize data availability patterns
def categorize_data_pattern(row):
    hometown = row['has_hometown_data']
    hs = row['has_hs_data']
    prev = row['has_prev_data']
    
    if not hometown and not hs and not prev:
        return "NO_DATA"
    elif not hometown and not hs and prev:
        return "PREV_ONLY"
    elif not hometown and hs and not prev:
        return "HS_ONLY"
    elif hometown and not hs and not prev:
        return "HOMETOWN_ONLY"
    elif not hometown and hs and prev:
        return "HS_PREV"
    elif hometown and not hs and prev:
        return "HOMETOWN_PREV"
    elif hometown and hs and not prev:
        return "HOMETOWN_HS"
    else:
        return "ALL_DATA"

final_round_df['data_pattern'] = final_round_df.apply(categorize_data_pattern, axis=1)

# Add team and season grouping
final_round_df['team_season'] = final_round_df['team'] + '_' + final_round_df['season'].astype(str)

print("\n📊 DATA PATTERN BREAKDOWN:")
pattern_counts = final_round_df['data_pattern'].value_counts()
for pattern, count in pattern_counts.items():
    print(f"  {pattern}: {count} players")

print(f"\n🏫 TEAM-SEASON BREAKDOWN:")
team_season_counts = final_round_df['team_season'].value_counts()
for team_season, count in team_season_counts.head(10).items():
    print(f"  {team_season}: {count} players")

print(f"\n📋 SAMPLE BY DATA PATTERN:")
# Show samples from each pattern
for pattern in pattern_counts.index:
    pattern_data = final_round_df[final_round_df['data_pattern'] == pattern]
    print(f"\n--- {pattern} ({len(pattern_data)} players) ---")
    display_cols = ['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']
    sample_size = min(5, len(pattern_data))
    print(pattern_data[display_cols].head(sample_size).to_string(index=False))

# Check for specific patterns that might be fixable
print(f"\n🔍 POTENTIAL PATTERNS TO INVESTIGATE:")

# Pattern 1: Previous school patterns
if len(final_round_df[final_round_df['has_prev_data']]) > 0:
    prev_schools = final_round_df[final_round_df['has_prev_data']]['previous_school'].value_counts()
    print(f"\nMost common previous schools:")
    for school, count in prev_schools.head(10).items():
        print(f"  {school}: {count} players")

# Pattern 2: High school patterns
if len(final_round_df[final_round_df['has_hs_data']]) > 0:
    hs_schools = final_round_df[final_round_df['has_hs_data']]['high_school'].value_counts()
    print(f"\nMost common high schools:")
    for school, count in hs_schools.head(10).items():
        print(f"  {school}: {count} players")

# Pattern 3: Hometown patterns that might have extractable info
if len(final_round_df[final_round_df['has_hometown_data']]) > 0:
    hometowns = final_round_df[final_round_df['has_hometown_data']]['hometown'].value_counts()
    print(f"\nMost common hometown entries:")
    for hometown, count in hometowns.head(10).items():
        print(f"  '{hometown}': {count} players")

print(f"\n💡 READY FOR FINAL CORRECTIONS!")
print(f"Use 'final_round_df' to analyze patterns and create targeted fixes.")
print(f"Current target: Get from 99.37% to 99.50%+ (need ~20 more corrections)")

final_round_df

🔍 SETTING UP FINAL ROUND ANALYSIS...
Remaining missing players: 98

📊 DATA PATTERN BREAKDOWN:
  NO_DATA: 72 players
  HOMETOWN_HS: 14 players
  HOMETOWN_ONLY: 11 players
  PREV_ONLY: 1 players

🏫 TEAM-SEASON BREAKDOWN:
  Bethune-Cookman_2011: 17 players
  Alabama A&M_2021: 8 players
  Alabama A&M_2020: 7 players
  Alabama State_2015: 6 players
  Southern_2024: 5 players
  Alabama A&M_2014: 4 players
  Grambling_2013: 4 players
  Alabama State_2016: 3 players
  Alabama A&M_2022: 3 players
  Florida A&M_2015: 3 players

📋 SAMPLE BY DATA PATTERN:

--- NO_DATA (72 players) ---
          name          team  season hometown high_school previous_school
   Darrel King Alabama State    2016      NaN         NaN            None
  Manny Holmes Alabama State    2016      NaN         NaN            None
  Torrey Haley Alabama State    2016      NaN         NaN            None
   Darrel King Alabama State    2015      NaN         NaN            None
Samuel Jackson Alabama State    2015      NaN     

,team,season,name,high_school,hometown,previous_school,class,team_state,player_state,is_international,has_hometown_data,has_hs_data,has_prev_data,data_pattern,team_season
1753,Alabama State,2016,Darrel King,NaN,NaN,None,Jr.,AL,None,False,False,False,False,NO_DATA,Alabama State_2016
1757,Alabama State,2016,Manny Holmes,NaN,NaN,None,Fr.,AL,None,False,False,False,False,NO_DATA,Alabama State_2016
1803,Alabama State,2016,Torrey Haley,NaN,NaN,None,Fr.,AL,None,False,False,False,False,NO_DATA,Alabama State_2016
1840,Alabama State,2015,Darrel King,NaN,NaN,None,Fr.,AL,None,False,False,False,False,NO_DATA,Alabama State_2015
1847,Alabama State,2015,Samuel Jackson,NaN,NaN,None,Fr.,AL,None,False,False,False,False,NO_DATA,Alabama State_2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15478,Bethune-Cookman,2011,Jeff Fagan,NaN,",",None,Rs.,FL,None,False,True,False,False,HOMETOWN_ONLY,Bethune-Cookman_2011
15480,Bethune-Cookman,2011,Darian McCaskill,NaN,",",None,Rs.,FL,None,False,True,False,False,HOMETOWN_ONLY,Bethune-Cookman_2011
15481,Bethune-Cookman,2011,John Powers,NaN,",",None,Rs.,FL,None,False,True,False,False,HOMETOWN_ONLY,Bethune-Cookman_2011
15482,Bethune-Cookman,2011,LeBrandon Richardson,NaN,",",None,So.,FL,None,False,True,False,False,HOMETOWN_ONLY,Bethune-Cookman_2011


In [28]:
# ===== FINAL ROUND: BETHUNE-COOKMAN SPECIFIC CORRECTIONS =====

print("Applying Final Round: Bethune-Cookman specific corrections...")
final_corrections = 0

# 1. Maurice Roberts for Bethune-Cookman in 2013 and 2014
mask = (swac_fb['name'] == 'Maurice Roberts') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'].isin([2013, 2014]))
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Immokalee, FL'
    swac_fb.loc[mask, 'high_school'] = 'Immokalee HS'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Maurice Roberts (2013,2014) → Immokalee, FL: {mask.sum()} corrections")

# 2. Johnathan Moment from Bethune-Cookman in 2013
mask = (swac_fb['name'] == 'Johnathan Moment') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2013)
if mask.any():
    swac_fb.loc[mask, 'high_school'] = 'Lake Highland Prep'
    swac_fb.loc[mask, 'hometown'] = 'Orlando, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Johnathan Moment → Orlando, FL: {mask.sum()} corrections")

# 3. Buddy Collins of Bethune-Cookman in 2011
mask = (swac_fb['name'] == 'Buddy Collins') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2011)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'DeLand, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Buddy Collins → DeLand, FL: {mask.sum()} corrections")

# 4. Daniel Jackson of Bethune-Cookman in 2011
mask = (swac_fb['name'] == 'Daniel Jackson') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2011)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Bartow, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Daniel Jackson → Bartow, FL: {mask.sum()} corrections")

# 5. Tavaris Bell of Bethune-Cookman in 2011
mask = (swac_fb['name'] == 'Tavaris Bell') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2011)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Jacksonville, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Tavaris Bell → Jacksonville, FL: {mask.sum()} corrections")

# 6. Ryan Davis of Bethune-Cookman in 2011
mask = (swac_fb['name'] == 'Ryan Davis') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2011)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Tampa, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Ryan Davis → Tampa, FL: {mask.sum()} corrections")

# 7. Jens Howe for Bethune-Cookman in 2014
mask = (swac_fb['name'] == 'Jens Howe') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2014)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Manti, UT'
    swac_fb.loc[mask, 'player_state'] = 'UT'
    final_corrections += mask.sum()
    print(f"✓ Fixed Jens Howe → Manti, UT: {mask.sum()} corrections")

# 8. Jawad Yatim for Bethune-Cookman in 2011
mask = (swac_fb['name'] == 'Jawad Yatim') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2011)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Shrewsbury, MA'
    swac_fb.loc[mask, 'player_state'] = 'MA'
    final_corrections += mask.sum()
    print(f"✓ Fixed Jawad Yatim → Shrewsbury, MA: {mask.sum()} corrections")

# 9. Xavier Reese for Bethune-Cookman in 2011
mask = (swac_fb['name'] == 'Xavier Reese') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2011)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Xavier Reese → FL: {mask.sum()} corrections")

# 10. Kyle Bailey of Bethune-Cookman in 2012
mask = (swac_fb['name'] == 'Kyle Bailey') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2012)
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'CA'
    final_corrections += mask.sum()
    print(f"✓ Fixed Kyle Bailey → CA: {mask.sum()} corrections")

# 11. Maurice Francois of Bethune-Cookman in 2011
mask = (swac_fb['name'] == 'Maurice Francois') & \
       (swac_fb['team'] == 'Bethune-Cookman') & \
       (swac_fb['season'] == 2011)
if mask.any():
    swac_fb.loc[mask, 'hometown'] = 'Palm Bay, FL'
    swac_fb.loc[mask, 'player_state'] = 'FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Maurice Francois → Palm Bay, FL: {mask.sum()} corrections")

# 12. Jazz Moss (any team/season with this name)
mask = (swac_fb['name'] == 'Jazz Moss')
if mask.any():
    swac_fb.loc[mask, 'player_state'] = 'FL'
    swac_fb.loc[mask, 'hometown'] = 'Ft. Lauderdale, FL'
    final_corrections += mask.sum()
    print(f"✓ Fixed Jazz Moss → Ft. Lauderdale, FL: {mask.sum()} corrections")

# Calculate final statistics
missing_after_final = swac_fb['player_state'].isnull().sum()
success_rate_after_final = ((len(swac_fb) - missing_after_final) / len(swac_fb) * 100)

print(f"\n🎯 FINAL ROUND CORRECTIONS SUMMARY:")
print(f"Final round corrections applied: {final_corrections}")
print(f"Missing count after Final: {missing_after_final}")
print(f"New success rate: {success_rate_after_final:.2f}%")
print(f"Improvement from Round 3: {missing_after_round3 - missing_after_final} fewer missing")

print(f"\n=== COMPLETE PROGRESS SUMMARY ===")
print(f"Starting success rate: 95.84%")
print(f"After city fixes: {final_success_rate_new:.2f}%")
print(f"After Round 1: {final_success_rate_after_irregularities:.2f}%") 
print(f"After Round 2: {success_rate_after_round2:.2f}%")
print(f"After Round 3: {success_rate_after_round3:.2f}%")
print(f"After Final: {success_rate_after_final:.2f}%")
print(f"Total improvement: {success_rate_after_final - 95.84:.2f} percentage points")

# Create updated missing players dataframe
missing_after_final_df = swac_fb[swac_fb['player_state'].isnull()].copy()
print(f"\n=== REMAINING MISSING PLAYERS ===")
print(f"Players still missing: {len(missing_after_final_df)}")

if len(missing_after_final_df) > 0:
    print(f"\nRemaining players sample:")
    print(missing_after_final_df[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].head(15).to_string(index=False))
else:
    print("🎉 PERFECT! ALL PLAYERS HAVE BEEN ASSIGNED STATES!")
    print(f"🏆 FINAL SUCCESS RATE: {success_rate_after_final:.2f}%")

Applying Final Round: Bethune-Cookman specific corrections...
✓ Fixed Maurice Roberts (2013,2014) → Immokalee, FL: 2 corrections
✓ Fixed Johnathan Moment → Orlando, FL: 1 corrections
✓ Fixed Buddy Collins → DeLand, FL: 1 corrections
✓ Fixed Daniel Jackson → Bartow, FL: 1 corrections
✓ Fixed Tavaris Bell → Jacksonville, FL: 1 corrections
✓ Fixed Ryan Davis → Tampa, FL: 1 corrections
✓ Fixed Jens Howe → Manti, UT: 2 corrections
✓ Fixed Jawad Yatim → Shrewsbury, MA: 1 corrections
✓ Fixed Xavier Reese → FL: 1 corrections
✓ Fixed Kyle Bailey → CA: 1 corrections
✓ Fixed Maurice Francois → Palm Bay, FL: 1 corrections
✓ Fixed Jazz Moss → Ft. Lauderdale, FL: 3 corrections

🎯 FINAL ROUND CORRECTIONS SUMMARY:
Final round corrections applied: 16
Missing count after Final: 85
New success rate: 99.45%
Improvement from Round 3: 13 fewer missing

=== COMPLETE PROGRESS SUMMARY ===
Starting success rate: 95.84%
After city fixes: 98.63%
After Round 1: 99.11%
After Round 2: 99.27%
After Round 3: 99.37%
Af